# Plot MPI Reproduction Results

This notebook reads `.npz` files written by the reproduction scripts under `scripts/results/` and saves a `plot.png` into each populated figure directory.

It detects the swept axis automatically:

- Fig. 3: physical error rate from `physicalNoiseArr`
- Fig. 4: ebit decoherence ratio from `transNoiseArr`
- Fig. 5: Pauli-string weight from `_wN` in the filename

Error bars use the same Bayes-factor binomial interval as `tmcbs.estimate_ler`.

In [1]:
from pathlib import Path
import re
import sys

import numpy as np
import matplotlib.pyplot as plt


def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "tmcbs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the TMCBS repository or scripts directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import tmcbs

RESULT_ROOT = ROOT / "scripts" / "results"
FIG_DIRS = {
    "fig3": RESULT_ROOT / "fig3",
    "fig4": RESULT_ROOT / "fig4",
    "fig5": RESULT_ROOT / "fig5",
}
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {ROOT}")
print(f"Result root:     {RESULT_ROOT}")

Repository root: /Users/jstack/Documents/tmcbs
Result root:     /Users/jstack/Documents/tmcbs/scripts/results


In [2]:
def yerr(ler, errs, shots):
    """Bayes-factor error bars as matplotlib offsets."""
    lo, hi = [], []
    for e, s, l in zip(errs, shots, ler):
        if s > 0 and np.isfinite(l):
            a, b = tmcbs.bayes_factor_interval(int(e), int(s))
            lo.append(l - a)
            hi.append(b - l)
        else:
            lo.append(0.0)
            hi.append(0.0)
    return [lo, hi]


def _load_curve(f, per_mode):
    with np.load(f, allow_pickle=True) as z:
        res = np.asarray(z["resultsArr"], float)
        errs = np.asarray(z["totalErrorsArr"], float)
        shots = np.asarray(z["numberOfShotsArr"], float)
        if per_mode:
            return (
                np.asarray(z["physicalNoiseArr"], float),
                res[:, 0],
                errs[:, 0],
                shots[:, 0],
            )
        return (
            np.asarray(z["transNoiseArr"], float),
            res[0, :],
            errs[0, :],
            shots[0, :],
        )


def _add_break_even(ax):
    lim = ax.get_xlim()
    diag = np.geomspace(*lim, 50)
    ax.plot(diag, diag, "k:", lw=1, label="break-even")
    ax.set_xlim(lim)


def _plot_fig3(files, src, colour):
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), sharey=True)
    panels = [("cnot_", "non-local CX"), ("tele_", "teleportation")]

    for ax, (prefix, title) in zip(axes, panels):
        for f in sorted(f for f in files if f.stem.startswith(prefix)):
            name = f.stem
            x, ler, e, s = _load_curve(f, per_mode=True)
            keep = np.isfinite(ler)
            if not np.any(keep):
                continue

            code = re.sub(r"^(cnot|tele)_", "", name)
            code = re.sub(r"_ebit(1|10)x$", "", code)
            ratio = "10x ebit" if "ebit10x" in name else "1x ebit"
            ls = ":" if "ebit10x" in name else "-"
            ax.errorbar(
                x[keep],
                ler[keep],
                yerr=yerr(ler[keep], e[keep], s[keep]),
                marker="o",
                ls=ls,
                color=colour(code),
                capsize=2,
                label=f"{code}, {ratio}",
            )

        ax.set(xscale="log", yscale="log", xlabel="physical error rate", title=title)
        _add_break_even(ax)
        ax.legend(fontsize=9, frameon=False)

    axes[0].set_ylabel("logical error rate")
    fig.suptitle(src.name)
    fig.tight_layout()
    return fig, axes


def plot_result_dir(src):
    """Plot all .npz results in one figure directory."""
    src = Path(src)
    files = sorted(src.glob("*.npz"))
    if not files:
        raise FileNotFoundError(f"No .npz files in {src}")

    cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    colors = {}

    def colour(series):
        return colors.setdefault(series, cycle[len(colors) % len(cycle)])

    if any(re.search(r"_w\d+", f.stem) for f in files):
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
        series = {}
        for f in files:
            name = f.stem
            match = re.search(r"_w(\d+)", name)
            if match is None:
                continue
            with np.load(f, allow_pickle=True) as z:
                ler = float(np.asarray(z["resultsArr"], float).ravel()[0])
                e = float(np.asarray(z["totalErrorsArr"], float).ravel()[0])
                s = float(np.asarray(z["numberOfShotsArr"], float).ravel()[0])
            if np.isfinite(ler):
                key = re.sub(r"_w\d+", "", name)
                series.setdefault(key, []).append((int(match.group(1)), ler, e, s))

        for key, pts in sorted(series.items()):
            pts.sort()
            w, ler, e, s = (np.array(c) for c in zip(*pts))
            ax.errorbar(w, ler, yerr=yerr(ler, e, s), marker="o",
                        color=colour(key), capsize=2, label=key)
        ax.set(yscale="log", xlabel="Pauli string weight", ylabel="logical error rate")
        ax.set_xticks(sorted({p[0] for pts in series.values() for p in pts}))
        ax.set_title(src.name)
        ax.legend(fontsize=11, frameon=False)
        fig.tight_layout()
        return fig, ax

    with np.load(files[0], allow_pickle=True) as first:
        per_mode = np.asarray(first["physicalNoiseArr"]).size > 1

    if src.name == "fig3" and per_mode:
        return _plot_fig3(files, src, colour)

    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for f in files:
        name = f.stem
        x, ler, e, s = _load_curve(f, per_mode)
        if per_mode:
            series = re.sub(r"_ebit(1|10)x$", "", name)
            ls = ":" if "ebit10x" in name else "-"
        else:
            series = re.sub(r"_(serial|line)$", "", name)
            ls = "--" if name.endswith("_line") else "-"

        keep = np.isfinite(ler)
        ax.errorbar(x[keep], ler[keep], yerr=yerr(ler[keep], e[keep], s[keep]),
                    marker="o", ls=ls, color=colour(series), capsize=2, label=name)

    ax.set(xscale="log", yscale="log", ylabel="logical error rate")
    if per_mode:
        _add_break_even(ax)
        ax.set_xlabel("physical error rate")
    else:
        ax.set_xlabel("ebit decoherence ratio (wait time / T1)")

    ax.set_title(src.name)
    ax.legend(fontsize=11, frameon=False)
    fig.tight_layout()
    return fig, ax


In [3]:
generated = []

for fig_name, src in FIG_DIRS.items():
    files = sorted(src.glob("*.npz"))
    if not files:
        print(f"Skipping {fig_name}: no .npz files in {src}")
        continue

    fig, ax = plot_result_dir(src)
    src.mkdir(parents=True, exist_ok=True)
    out = src / "plot.png"
    fig.savefig(out, dpi=140)
    generated.append(out)
    print(f"Wrote {out}")
    plt.show()

generated

Skipping fig3: no .npz files in /Users/jstack/Documents/tmcbs/scripts/results/fig3
Skipping fig4: no .npz files in /Users/jstack/Documents/tmcbs/scripts/results/fig4
Skipping fig5: no .npz files in /Users/jstack/Documents/tmcbs/scripts/results/fig5


[]